## **Profile a simple CNN trained on synthetic data, with nvtx library. Example with optimizations applied**
Current notebook has some code fixes to optimize/resolve the bottlenecks observed in earlier run, i.e. as part of the notebook `debug-llm-bottlenecks-with-nvidia-nsight/lab-01/01-simple-cnn-with-nvtx/simple-cnn-with-nvtx-baseline.ipynb` .

Just select the cell you want to profile and run with nsight profiler (green run button from the toolbar). If you like to profile a portion of code, split the code into multiple cells and profile the cell you like to investigate.<br/>
As an alternate you could also add **nvtx** annotations for the code snippet you want to dig deeper. Let's experiment with **nvtx** library in this notebook.
<br/> - **[NVTX documentation](https://nvidia.github.io/NVTX/)**

## Key Optimizations:

**Memory Optimizations:**
• model.half(): FP16 precision for 2x memory savings
• Pre-move data to GPU: data.to(device) before training
• torch.cuda.empty_cache(): Clean memory before profiling

**DataLoader Optimizations:**
• num_workers=4: Multi-threaded data loading
• pin_memory=True: Faster CPU-GPU transfers
• persistent_workers=True: Reuse worker processes
• Larger batch_size=128: Better GPU utilization

**Training Loop Optimizations:**
• Removed loss.item() calls during training (no CPU-GPU sync)
• Data already on GPU (no transfer overhead)
• Less frequent printing to avoid synchronization

**Profiling Optimizations:**
• Clean memory state before profiling
• Clear synchronization boundaries
• Warm-up run to stabilize GPU state
<br/>
### Let's get started !

In [ ]:
!pip install matplotlib torch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import nvtx

# Clear cache
torch.cuda.empty_cache()

class SimpleCNN(nn.Module):
    def __init__(self):
        nvtx.push_range("model init", domain="nvtx-custom")
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        nvtx.pop_range(domain="nvtx-custom")
        
    def forward(self, x):
        nvtx.push_range("model forward pass", domain="nvtx-custom")
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 256 * 4 * 4)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        nvtx.pop_range(domain="nvtx-custom")
        return x

nvtx.push_range("model initialization and loading", domain="nvtx-custom")
# Setup for optimized profiling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

# Use mixed precision for faster training
model = model.half()  # FP16 for better performance

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
nvtx.pop_range(domain="nvtx-custom")

nvtx.push_range("data preparation", domain="nvtx-custom")
# Create data once and move to GPU
batch_size = 128  # Larger batch size for better GPU utilization
data = torch.randn(5000, 3, 32, 32, dtype=torch.float16).to(device)  # Pre-move to GPU
targets = torch.randint(0, 10, (5000,)).to(device)  # Pre-move to GPU


dataset = TensorDataset(data, targets)

# Optimized DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=0,      # Multi-threaded data loading
)
nvtx.pop_range(domain="nvtx-custom")

# Optimized training loop
def train_step():
    model.train()
    
    # Remove synchronous operations during training
    for batch_idx, (batch_data, batch_target) in enumerate(dataloader):
        nvtx_msg = "batch "+ str(batch_idx)
        nvtx.push_range(nvtx_msg, domain="nvtx-custom")
        optimizer.zero_grad()
        output = model(batch_data)
        nvtx.push_range("backward pass - "+nvtx_msg, domain="nvtx-custom")
        nvtx.push_range("loss computation", domain="nvtx-custom")
        loss = criterion(output, batch_target)
        loss.backward()
        nvtx.pop_range(domain="nvtx-custom")
        nvtx.push_range("optimizer step", domain="nvtx-custom")
        optimizer.step()
        nvtx.pop_range(domain="nvtx-custom")
        nvtx.pop_range(domain="nvtx-custom")
        nvtx.pop_range(domain="nvtx-custom")
        # Only print occasionally to avoid synchronization
        if batch_idx % 5 == 0:
            print(f"Batch {batch_idx}")  # No loss.item() to avoid sync
        
        # if batch_idx >= 10:  # [optional] just a few batches
        #     break
        
print("Starting profiled training...")
torch.cuda.synchronize()  
train_step()  
torch.cuda.synchronize()  
print("Profiling complete!")

## **Sample Profiling Reports**

![](../../static-assets/lab-01/simple-cnn-with-nvtx-optimized.png)

![](../../static-assets/lab-01/simple-cnn-with-nvtx-optimized-gpu-util.png)

![](../../static-assets/lab-01/simple-cnn-with-nvtx-optimized-gpu-gaps.png)

![](../../static-assets/lab-01/simple-cnn-with-nvtx-optimized-DtoH.png)

![](../../static-assets/lab-01/simple-cnn-with-nvtx-optimized-memops.png)